# Running SKIRT simulations

### Prerequisites
Before running a simulation, make sure you have installed:
1. **SKIRT and PTS** (Python Toolkit for SKIRT) — see the [official SKIRT installation guide](https://skirt.ugent.be/root/_installation_guide.html).
2. **`SKIRT-tools` and its Python dependencies** (`pip install -e .` from the repository root).

### Initialize a simulation
To run a radiative transfer simulation, initialize a `Galaxy` object from your FIRE snapshot and pass it to `Sim` along with an output directory path:

In [1]:

import sys
sys.path.append("../src")

import SKIRT_tools
import SKIRT_tools.skirt_tools as sktools

rundir = "/nfspool-0/home/jennydw/SKIRT-tools/tutorials/example" # Absolute path to the simulation directory
# load galaxy from .hdf5 file
m12f = sktools.Galaxy("m12f_mass56000_cr_700", hdf5_file = "m12f_mass56000_cr_700.h5")
# input the simulation parameters you want
m12f_simulation = sktools.Sim(m12f, rundir, n_photon_packets=1e3)
# As a first test, we initialize the simulation with only 1000 photon packets.


Note: No snapshot number found.


`Sim` only requires `galaxy` and `rundir`, but you can pass additional arguments to change parameters:

| Parameter | Type | Description |
| :--- | :--- | :--- |
| `max_radius` | `int` / `float` | Maximum radial distance (in kpc) for star and gas particles. Particles outside this radius are ignored. |
| `import_velocity` | `bool` | Include Doppler shifts from particle velocities relative to the model frame. Requires a `"velocity"` column in `star_cols` and `gas_cols`. |
| `import_gas_temp` | `bool` | Export gas temperature tables so SKIRT can apply a maximum gas temperature threshold for dust survival. Requires a `"temperature"` column in `gas_cols`. |
| `n_random_projections` | `int` | Number of randomly sampled viewing angles generated in addition to the default `xy`, `xz`, and `yz` projections. |
| `n_photon_packets` | `int` | Number of photon packets launched per wavelength grid bin. The more photon packets, the more computationally expensive; A rule of thumb is 50 photon packets per pixel.
 |

Once we are happy with our parameters, we use `build()` to write the necessary data files and parameter files that SKIRT requires for a simulation, and outputs them in the run directory.

- `star.scol`: column file for star data
- `gas.scol`: column file for gas data
- `*.ski`: parameter file for the SKIRT simulation

This only needs to be done once; if `build()` detects that these file names are already in the run directory, it will skip creating them. If you want to force these files to be created, for example if you updated the data, you can pass `force_make_scol = True` and/or `force_make_ski = True`.

In [2]:

# creates a run directory and initializes parameter files
m12f_simulation.build(force_make_scol=False, force_make_ski=False)


All parameter and data files already exist in run directory.


By default, running a simulation with `Sim` will output three orthogonal projections: `xy`, `yz`, and `zx`, relative to the coordinates that are passed into `Galaxy`:
- `xy` is unrotated
- `yz` is the 
- `xz`

However, you can add as many instruments, corresponding to projections, as you want. For example, if you wanted to add a random projection:

In [3]:

import SKIRT_tools.observation_utils as ou

inclination, azimuth, roll = ou.random_viewing_angles()
m12f_simulation.add_instrument("rand1", inclination, azimuth, roll)


### Run a simulation

After creating the particle data and parameter files, the simulation is ready to be run with the following code.

You might need to copy this script into a `.py` file and run it with the right environment and dependencies.

In [ ]:

m12f_simulation.run(nthreads=4, nprocesses=2)


We can also run the simulation without initalizing a `Galaxy` or `Sim` object. To do this, `rundir` should already have the relevant the particle data (`.scol`) and parameter (`.ski`) files.

In [ ]:

sktools.run_simulation(name="m12f_f", rundir=rundir, nthreads=4, nprocesses=2)


Lastly, we can run the simulation directly using SKIRT.

To run the simulation, navigate to your run directory and execute the following skirt command.

    cd /nfspool-0/home/jennydw/SKIRT-tools/tutorials/example
    skirt m12f_mass56000_cr_700.ski

You may need to load the gnu module if you encounter an error on greenplanet:

    ml gnu/11.3.0